Project Overview: Telecom Customer Churn Prediction
Objective:
Predict customer churn (cancellation) using historical data to identify retention opportunities.

Technical Approach
Data Analysis:

Cleaned and preprocessed the dataset using SQL (Snowflake), Python (Pandas, Seaborn, Scikit-Learn).

Model Building & Validation:

Selected Random Forest classifier for its robustness and interpretability.

Optimised hyperparameters (n_estimators, max_depth, min_samples_split) using GridSearchCV.

Split data into training and validation sets.

Evaluated model performance using precision, recall, and ROC-AUC metrics.

Visualised results with confusion matrices, ROC curves, and precision-recall curves.

Project Outcomes
Model Performance:

Achieved a strong ROC-AUC of 0.83, demonstrating excellent ability to rank customers by churn risk.

Business Impact:

Provided clear, actionable insights to guide retention strategies and reduce customer churn.

Technical Deliverables:

Cleaned and preprocessed dataset.

Optimised predictive model.

Visualisations and business reports.

In [ ]:
SELECT * FROM PIXAR.PUBLIC.TELECOM_CUSTOMER_CHURN;

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session
from sklearn.model_selection import train_test_split

# Get active session
session = get_active_session()

# Load data
df = session.sql("SELECT * FROM PIXAR.PUBLIC.TELECOM_CUSTOMER_CHURN").to_pandas()

# Preprocessing
df['TOTALCHARGES'] = pd.to_numeric(df['TOTALCHARGES'], errors='coerce')
df.dropna(subset=['TOTALCHARGES'], inplace=True)

# Identify ALL categorical columns (excluding CUSTOMERID and CHURN)
categorical_cols = [
    'GENDER', 'SENIORCITIZEN', 'PARTNER', 'DEPENDENTS', 
    'PHONESERVICE', 'MULTIPLELINES', 'INTERNETSERVICE', 
    'ONLINESECURITY', 'ONLINEBACKUP', 'DEVICEPROTECTION', 
    'TECHSUPPORT', 'STREAMINGTV', 'STREAMINGMOVIES', 
    'PAPERLESSBILLING', 'CONTRACT', 'PAYMENTMETHOD'
]

# Convert boolean-like columns to categorical
for col in ['SENIORCITIZEN', 'PARTNER', 'DEPENDENTS', 'PAPERLESSBILLING']:
    df[col] = df[col].astype(str)

# Apply one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Prepare features and target
X = df.drop(['CUSTOMERID', 'CHURN'], axis=1)
y = df['CHURN'].astype(int)

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify
print(f"Cleaned data shape: {df.shape}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Churn distribution:\n{y.value_counts()}")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# Train model
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)

# Predict and validate
y_pred_logreg = logreg.predict(X_val)
y_proba_logreg = logreg.predict_proba(X_val)[:, 1]

# Metrics
precision_logreg = precision_score(y_val, y_pred_logreg)
recall_logreg = recall_score(y_val, y_pred_logreg)
roc_auc_logreg = roc_auc_score(y_val, y_proba_logreg)

print("Logistic Regression Results:")
print(f"Precision: {precision_logreg:.3f}")
print(f"Recall: {recall_logreg:.3f}")
print(f"ROC-AUC: {roc_auc_logreg:.3f}")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define model and parameter grid
rf = RandomForestClassifier(random_state=42)
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

# Grid search
grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='roc_auc')
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

# Predict and validate
y_pred_rf = best_rf.predict(X_val)
y_proba_rf = best_rf.predict_proba(X_val)[:, 1] # Probability scores for positive class

# Metrics
precision_rf = precision_score(y_val, y_pred_rf)
recall_rf = recall_score(y_val, y_pred_rf)
roc_auc_rf = roc_auc_score(y_val, y_proba_rf)

print("\nRandom Forest Results:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Precision: {precision_rf:.3f}")
print(f"Recall: {recall_rf:.3f}")
print(f"ROC-AUC: {roc_auc_rf:.3f}")


In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt
# 1. Confusion Matrix
cm = confusion_matrix(y_val, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Churned', 'Churned'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

# Top-left: True Negatives (correct non-churn predictions)
# Bottom-right: True Positives (correct churn predictions)
# Top-right: False Positives (Type I errors)
# Bottom-left: False Negatives (Type II errors)

In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt
# 2. ROC Curve
fpr, tpr, thresholds = roc_curve(y_val, y_proba_rf)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc_rf:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

# Curve closer to top-left = better performance
# AUC > 0.8 indicates strong discrimination

In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt 
# 3. Precision-Recall Curve
precision, recall, thresholds = precision_recall_curve(y_val, y_proba_rf)
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()

# Crucial for imbalanced datasets
# Higher curve = better performance
# Shows precision-recall tradeoff

What This Curve Shows
The Classic Precision-Recall Trade-off:

Left side (low recall, high precision): Model is very conservative - when it predicts churn, it's usually correct, but it misses many actual churners

Right side (high recall, low precision): Model catches most churners but generates many false alarms


Key Insights
Sweet Spot Identification:
Optimal operating point: Around 0.1-0.2 recall, 0.8-0.85 precision
At this point, you maintain high accuracy (85% of churn predictions are correct) whilst capturing 10-20% of actual churners

Performance Characteristics:
Steep decline after 0.3 recall: Indicates diminishing returns - catching more churners comes at high cost of false alarms

Baseline precision (~0.27): Suggests roughly 27% of customers churn (class imbalance)

Business Recommendations
1. Tiered Retention Strategy
Recall Range	Precision	Business Action	Resource Allocation
0.0-0.2	0.8-0.9	Premium retention offers	High-value, personalised
0.2-0.4	0.7-0.8	Standard retention campaigns	Moderate cost interventions
0.4+	<0.7	Automated engagement	Low-cost, mass communication

In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt
# 4. Feature Importance
feature_importances = pd.Series(best_rf.feature_importances_, index=X_train.columns)
top_features = feature_importances.nlargest(10)
plt.figure(figsize=(10, 6))
top_features.sort_values().plot(kind='barh')
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance Score')
plt.show()

# Identifies key drivers of churn predictions
# Helps prioritize business actions

Recommended Priorities

- Review and enhance fibre optic service offerings to address potential pain points.
- Promote more stable payment methods to customers using electronic cheques.
- Encourage longer-term contracts with targeted incentives.
- Upsell online security and tech support services to at-risk customers.

In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt
import numpy as np

# 5. Threshold Analysis (Optional)
# Shows how precision/recall change with different classification thresholds
thresholds = np.arange(0.1, 1.0, 0.1)
precisions = []
recalls = []

for thresh in thresholds:
    y_pred_thresh = (y_proba_rf >= thresh).astype(int)
    precisions.append(precision_score(y_val, y_pred_thresh))
    recalls.append(recall_score(y_val, y_pred_thresh))

plt.figure(figsize=(8, 6))
plt.plot(thresholds, precisions, 'b-', label='Precision')
plt.plot(thresholds, recalls, 'g-', label='Recall')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision-Recall Tradeoff')
plt.legend()
plt.show()